In [1]:
import polars as pl

In [2]:
inicial = (
    pl.scan_parquet(r"C:\Users\diogo.durao\Documents\covid_sp.parquet")
    .select([
        "paciente_idade",
        "paciente_enumSexoBiologico",
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "vacina_dataAplicacao",
        "vacina_fabricante_nome",
        "vacina_categoria_nome"
    ])
    .with_columns([
        pl.col("paciente_idade").cast(pl.Int64, strict=False),
        pl.col("vacina_dataAplicacao").str.to_date("%Y-%m-%d", strict=False)
    ])
    .filter(
        pl.col("paciente_idade").is_not_null() &
        pl.col("paciente_endereco_nmMunicipio").is_not_null() &
        pl.col("paciente_endereco_uf").is_not_null()
    )
)

In [4]:
display(inicial.collect().head(10))

paciente_idade,paciente_enumSexoBiologico,paciente_endereco_nmMunicipio,paciente_endereco_uf,vacina_dataAplicacao,vacina_fabricante_nome,vacina_categoria_nome
i64,str,str,str,date,str,str
32,"""F""","""GUARULHOS""","""SP""",2022-04-05,"""JANSSEN""","""Comorbidades"""
59,"""F""","""IBITINGA""","""SP""",2021-05-31,"""ASTRAZENECA/FIOCRUZ""","""Faixa Etária"""
85,"""M""","""SAO PAULO""","""SP""",2021-12-04,"""PFIZER""","""Faixa Etária"""
35,"""M""","""SAO PAULO""","""SP""",2021-08-14,"""SINOVAC/BUTANTAN""","""Faixa Etária"""
53,"""M""","""SAO PAULO""","""SP""",2021-09-14,"""PFIZER""","""Faixa Etária"""
43,"""M""","""LEME""","""SP""",2021-06-30,"""PFIZER""","""Faixa Etária"""
14,"""F""","""SAO JOSE DOS CAMPOS""","""SP""",2021-11-09,"""PFIZER""","""Faixa Etária"""
58,"""F""","""RANCHARIA""","""SP""",2022-07-26,"""ASTRAZENECA/FIOCRUZ""","""Pessoas com Deficiência"""
27,"""F""","""ALUMINIO""","""SP""",2021-10-19,"""PFIZER""","""Trabalhadores de Saúde"""


In [5]:
df_municipios = (
    inicial
    .group_by(
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf"
    )
    .agg([
        pl.len().alias("Total_Vacinacoes"),

        pl.col("paciente_idade")
        .mean()
        .round(2)
        .alias("Media_Idade")
    ])
)

In [6]:
df_agregado = df_municipios.collect()

In [7]:
# 3. Calcular quartis

q1 = (
    df_agregado
    .select(
        pl.col("Total_Vacinacoes")
        .quantile(0.25)
    )
    .item()
)

q3 = (
    df_agregado
    .select(
        pl.col("Total_Vacinacoes")
        .quantile(0.75)
    )
    .item()
)

print(f"Municípios abaixo de {q1:.0f} vacinações são 'Baixa Vacinação'")
print(f"Municípios acima de {q3:.0f} vacinações são 'Alta Vacinação'")

Municípios abaixo de 27 vacinações são 'Baixa Vacinação'
Municípios acima de 499 vacinações são 'Alta Vacinação'


In [8]:
# 4. Criar classificação

df_export = (
    df_agregado
    .with_columns(
        pl.when(
            pl.col("Total_Vacinacoes") > q3
        )
        .then(pl.lit("Alta Vacinação"))
        
        .when(
            pl.col("Total_Vacinacoes") < q1
        )
        .then(pl.lit("Baixa Vacinação"))
        
        .otherwise(pl.lit("Média Vacinação"))
        
        .alias("Etiqueta_Classificacao")
    )
)

In [9]:
# 5. Exportar CSV

df_export.write_csv("vacinas_sp_powerbi.csv")

print("Arquivo 'vacinas_sp_powerbi.csv' gerado com sucesso!")

Arquivo 'vacinas_sp_powerbi.csv' gerado com sucesso!
